In [ ]:
!pip install accelerate

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config
from tqdm import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
config = GPT2Config()
model = GPT2LMHeadModel(config)

In [ ]:
with open('bible.txt', 'r') as file:
    bible_text = file.read()

In [ ]:
chunk_size = 1024
bible_chunks = [bible_text[i:i+chunk_size] for i in range(0, len(bible_text), chunk_size)]

In [ ]:
tokens = [tokenizer(chunk, return_tensors='pt', max_length=chunk_size, truncation=True, padding='max_length') for chunk in bible_chunks]

In [ ]:
tokens = [{'input_ids': t['input_ids'].to(device), 'attention_mask': t['attention_mask'].to(device)} for t in tokens]

In [ ]:
model = model.to(device)
model.train()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [ ]:
patience = 2
best_loss = float('inf')
patience_counter = 0

In [ ]:
epochs = 1
total_steps = epochs * len(tokens)
step = 0

for epoch in range(epochs):
    total_loss = 0
    start_time = time.time()
    with tqdm(total=len(tokens), desc=f'Epoch {epoch + 1}/{epochs}') as pbar:
        for batch in tokens:
            step += 1
            optimizer.zero_grad()

            outputs = model(**batch, labels=batch['input_ids'])
            loss = outputs.loss
            loss.backward()

            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
            pbar.update(1)

    avg_loss = total_loss / len(tokens)
    end_time = time.time()
    epoch_time = end_time - start_time
    print(f"Epoch {epoch + 1}/{epochs} completed in {epoch_time:.2f} seconds. Average Loss: {avg_loss:.4f}")


    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping triggered")
        break


Epoch 1/1: 100%|██████████| 4457/4457 [27:51<00:00,  2.67it/s, loss=0.838]

Epoch 1/1 completed in 1671.23 seconds. Average Loss: 1.1845


In [ ]:
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
test_questions = ["Who led the Israelites out of Egypt?", "What is the last book of the Bible?"]
for question in test_questions:
    input_ids = tokenizer.encode(question, return_tensors='pt').to(device)
    outputs = model.generate(input_ids, max_length=50)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Q: {question}\nA: {response}\n")

Q: Who led the Israelites out of Egypt?
A: Who led the Israelites out of Egypt?
Revelation 21:2	And the earth, and the earth, and the earth, and the earth, and the earth, and the earth, and the earth, and the earth.


Q: What is the last book of the Bible?
A: What is the last book of the Bible?
Revelation 21:2	And the earth, and the earth, and the earth, and the earth, and the earth, and the earth, and the earth, and the earth.




In [ ]:
model.save_pretrained('bible_gpt2')
tokenizer.save_pretrained('bible_gpt2')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bible_gpt2/tokenizer_config.json', 'bible_gpt2/tokenizer.json')